# Setup & Load Master Data

In [3]:
import re
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process

engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

kodepos_df = pd.read_sql("SELECT kode, kodepos FROM wilayah_kodepos.wilayah_kodepos", engine)
kodepos_df["kodepos"] = kodepos_df["kodepos"].astype(str).str.strip()

print(wilayah.shape, kodepos_df.shape)
wilayah.head()

(91599, 2) (83762, 2)


,kode,nama
0,11,ACEH
1,11.01,KABUPATEN ACEH SELATAN
2,11.01.01,BAKONGAN
3,11.01.01.2001,KEUDE BAKONGAN
4,11.01.01.2002,UJONG MANGKI


# Get Level & Split Per Level

In [4]:
def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)
df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"provinsi={len(df_prov)}, kabkota={len(df_kabkota)}, kecamatan={len(df_kec)}, desa={len(df_desa)}")

provinsi=38, kabkota=514, kecamatan=7285, desa=83762


# Core Name

In [5]:
ADMIN_WORDS = {"KOTA", "KABUPATEN", "KAB", "ADMINISTRASI", "ADM"}

def core_name(nama):
    tokens = nama.replace(".", " ").split()
    return " ".join(t for t in tokens if t not in ADMIN_WORDS).strip()

for df_ in (df_prov, df_kabkota, df_kec, df_desa):
    df_["nama_core"] = df_["nama"].apply(core_name)

# test cepat
print(core_name("KOTA ADMINISTRASI JAKARTA SELATAN"))
df_kabkota[df_kabkota["nama"].str.contains("JAKARTA SELATAN")][["nama", "nama_core"]]

JAKARTA SELATAN


,nama,nama_core
158,KOTA ADMINISTRASI JAKARTA SELATAN,JAKARTA SELATAN


# Dictionary

In [6]:
PROVINCE_ALIASES = {
    "DKI JAKARTA": "DKI JAKARTA", "DKI": "DKI JAKARTA", "JKT": "DKI JAKARTA",
    "JAWA BARAT": "JAWA BARAT", "JABAR": "JAWA BARAT",
    "JAWA TENGAH": "JAWA TENGAH", "JATENG": "JAWA TENGAH",
    "JAWA TIMUR": "JAWA TIMUR", "JATIM": "JAWA TIMUR",
    "KALIMANTAN SELATAN": "KALIMANTAN SELATAN", "KALSEL": "KALIMANTAN SELATAN",
    "KALIMANTAN TIMUR": "KALIMANTAN TIMUR", "KALTIM": "KALIMANTAN TIMUR",
    "KALIMANTAN BARAT": "KALIMANTAN BARAT", "KALBAR": "KALIMANTAN BARAT",
    "KALIMANTAN TENGAH": "KALIMANTAN TENGAH", "KALTENG": "KALIMANTAN TENGAH",
    "KALIMANTAN UTARA": "KALIMANTAN UTARA", "KALTARA": "KALIMANTAN UTARA",
    "SUMATERA UTARA": "SUMATERA UTARA", "SUMUT": "SUMATERA UTARA",
    "SUMATERA BARAT": "SUMATERA BARAT", "SUMBAR": "SUMATERA BARAT",
    "SUMATERA SELATAN": "SUMATERA SELATAN", "SUMSEL": "SUMATERA SELATAN",
    "SULAWESI SELATAN": "SULAWESI SELATAN", "SULSEL": "SULAWESI SELATAN",
    "SULAWESI UTARA": "SULAWESI UTARA", "SULUT": "SULAWESI UTARA",
    "SULAWESI TENGAH": "SULAWESI TENGAH", "SULTENG": "SULAWESI TENGAH",
    "SULAWESI TENGGARA": "SULAWESI TENGGARA", "SULTRA": "SULAWESI TENGGARA",
    "SULAWESI BARAT": "SULAWESI BARAT", "SULBAR": "SULAWESI BARAT",
    "NUSA TENGGARA BARAT": "NUSA TENGGARA BARAT", "NTB": "NUSA TENGGARA BARAT",
    "NUSA TENGGARA TIMUR": "NUSA TENGGARA TIMUR", "NTT": "NUSA TENGGARA TIMUR",
    "KEPULAUAN RIAU": "KEPULAUAN RIAU", "KEPRI": "KEPULAUAN RIAU",
    "KEPULAUAN BANGKA BELITUNG": "KEPULAUAN BANGKA BELITUNG", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DAERAH ISTIMEWA YOGYAKARTA": "DAERAH ISTIMEWA YOGYAKARTA",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA", "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPUA BARAT": "PAPUA BARAT", "PAPBAR": "PAPUA BARAT",
}
CITY_ALIASES = {
    "JAKARTA SELATAN": "JAKARTA SELATAN", "JAKSEL": "JAKARTA SELATAN",
    "JAKARTA PUSAT": "JAKARTA PUSAT", "JAKPUS": "JAKARTA PUSAT",
    "JAKARTA UTARA": "JAKARTA UTARA", "JAKUT": "JAKARTA UTARA",
    "JAKARTA BARAT": "JAKARTA BARAT", "JAKBAR": "JAKARTA BARAT",
    "JAKARTA TIMUR": "JAKARTA TIMUR", "JAKTIM": "JAKARTA TIMUR",
    "BANDUNG": "BANDUNG", "BDG": "BANDUNG", "SURABAYA": "SURABAYA", "SBY": "SURABAYA",
    "SEMARANG": "SEMARANG", "SMG": "SEMARANG", "MEDAN": "MEDAN", "MDN": "MEDAN",
    "MAKASSAR": "MAKASSAR", "MKS": "MAKASSAR", "PALEMBANG": "PALEMBANG", "PLB": "PALEMBANG",
    "TANGERANG": "TANGERANG", "TGRG": "TANGERANG", "TNGRG": "TANGERANG",
    "BEKASI": "BEKASI", "BKS": "BEKASI", "DEPOK": "DEPOK", "DPK": "DEPOK",
    "BOGOR": "BOGOR", "BGR": "BOGOR",
}
_ALL_ALIASES = {**CITY_ALIASES, **PROVINCE_ALIASES}
_ALIAS_KEYS_SORTED = sorted(_ALL_ALIASES.keys(), key=len, reverse=True)
_ALIAS_PATTERN = re.compile(r"\b(" + "|".join(re.escape(k) for k in _ALIAS_KEYS_SORTED) + r")\b")

def expand_abbreviations(text):
    return _ALIAS_PATTERN.sub(lambda m: _ALL_ALIASES[m.group(1)], text)

# test
print(expand_abbreviations("JAKSEL DAN JATENG"))

JAKARTA SELATAN DAN JAWA TENGAH


# Pattern Structural

In [105]:
# Pattern Build

STREET_PATTERN   = r"\b(JALAN|JL|JLN)\b\.?\s*"
BUILDING_PATTERN = r"\b(GEDUNG|GD|GDG|GED|MENARA|MNR|TOWER|TWR|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b\.?"
FLOOR_PATTERN    = r"\b(LANTAI|LT|LTAI|FLOOR|FLR)\b\.?\s*([A-Z0-9\-]+)"
NO_PATTERN       = r"\b(NOMOR|NMR|NO)\b\.?\s*([A-Z0-9\-\/]+)"
BLOK_PATTERN     = r"\bBLOK\b\.?\s*([A-Z0-9\-]+)"
KAV_PATTERN      = r"\bKAV(?:LING)?\b\.?\s*([A-Z0-9\-]+)"
POSTAL_PATTERN   = r"(?<!\d)(\d{5})(?!\d)"
RT_RW_PATTERNS = [
    r"\bRT\s*/?\s*RW\b\.?\s*(\d+)\s*/\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\s*/\s*RW\b\.?\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\D{0,15}RW\b\.?\s*(\d+)",
    r"\bRT\b\.?\s*(\d+)\s*/\s*(\d+)\b",
]

In [106]:
def tokenize(text):
    return re.findall(r"[A-Z0-9]+", text)

In [107]:
remaining = "JL BLOK M NO. 12 LT 17 BLOK KV, RT 02RW 07 GD SENTRAYA MARYA JAKARTA 12950- INDONESIA , JAKARTA SELATAN".upper()

def extract_and_remove_kodepos(text):
    m = re.search(POSTAL_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

kodepos_hasil, remaining = extract_and_remove_kodepos(remaining)
print("kodepos:", kodepos_hasil)
print("remaining:", remaining)

kodepos: 12950
remaining: JL BLOK M NO. 12 LT 17 BLOK KV, RT 02RW 07 GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [108]:
def extract_and_remove_rt_rw(text):
    for p in RT_RW_PATTERNS:
        m = re.search(p, text)
        if m:
            rt, rw = m.group(1), m.group(2)
            new_text = text[:m.start()] + text[m.end():]
            return rt, rw, new_text
    return None, None, text

rt_hasil, rw_hasil, remaining = extract_and_remove_rt_rw(remaining)
print("rt:", rt_hasil, "| rw:", rw_hasil)
print("remaining:", remaining)

rt: 02 | rw: 07
remaining: JL BLOK M NO. 12 LT 17 BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [109]:
def extract_and_remove_no(text):
    m = re.search(NO_PATTERN, text)
    if not m:
        return None, text
    value = m.group(2)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

no_hasil, remaining = extract_and_remove_no(remaining)
print("no:", no_hasil)
print("remaining:", remaining)

no: 12
remaining: JL BLOK M  LT 17 BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [110]:
def extract_and_remove_lantai(text):
    m = re.search(FLOOR_PATTERN, text)
    if not m:
        return None, text
    value = m.group(2)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

lantai_hasil, remaining = extract_and_remove_lantai(remaining)
print("lantai:", lantai_hasil)
print("remaining:", remaining)

lantai: 17
remaining: JL BLOK M   BLOK KV,  GD SENTRAYA MARYA JAKARTA - INDONESIA , JAKARTA SELATAN


In [111]:
known_wilayah_tokens = set(tokenize(" ".join(df_prov["nama_core"]))) | set(tokenize(" ".join(df_kabkota["nama_core"])))
struktural_words = {"NO", "NOMOR", "NMR", "LT", "LANTAI", "LTAI", "FLOOR", "FLR", "RT", "RW", "BLOK", "KAV", "KAVLING"}

def extract_and_remove_gedung(text):
    pattern = re.compile(
        r"\b(?:GEDUNG|GD|GDG|GED|MENARA|MNR|TOWER|TWR|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b\.?\s*"
        r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,2})"
    )
    m = pattern.search(text)
    if not m:
        return None, text

    words = m.group(1).split()
    cut_index = len(words)
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:
            cut_index = min(cut_index, i)
            break
    for i, w in enumerate(words[:cut_index]):
        if w in struktural_words:
            cut_index = min(cut_index, i)
            break

    nama = " ".join(words[:cut_index]).strip(" .,-")
    if not nama:
        return None, text

    # hapus SELURUH match keyword+nama dari teks (bukan cuma nama-nya)
    full_match_text = m.group(0)
    consumed = full_match_text[:len(m.group(0)) - len(m.group(1))] + nama
    new_text = text.replace(consumed, "", 1)
    return nama, new_text

gedung_hasil, remaining = extract_and_remove_gedung(remaining)
print("gedung:", gedung_hasil)
print("remaining:", remaining)

gedung: SENTRAYA MARYA
remaining: JL BLOK M   BLOK KV,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [113]:
def extract_and_remove_jalan(text):
    pattern = re.compile(
        r"\b(?:JALAN|JL|JLN)\b\.?\s*"
        r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,3})"
    )
    m = pattern.search(text)
    if not m:
        return None, text

    words = m.group(1).split()

    cut_index = len(words)

    # cek known wilayah tokens -- boleh cut di index manapun, termasuk index 0
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:
            cut_index = min(cut_index, i)
            break

    # cek struktural words -- SKIP index 0, karena kata pertama boleh jadi bagian nama asli
    # (misal "Blok M" -- "BLOK" di index 0 itu sah, bukan penanda field terpisah)
    for i, w in enumerate(words[:cut_index]):
        if i == 0:
            continue
        if w in struktural_words:
            cut_index = min(cut_index, i)
            break

    nama = " ".join(words[:cut_index]).strip(" .,-")
    if not nama:
        return None, text

    full_match_text = m.group(0)
    consumed = full_match_text[:len(m.group(0)) - len(m.group(1))] + nama
    new_text = text.replace(consumed, "", 1)
    return nama, new_text

jalan_hasil, remaining = extract_and_remove_jalan(remaining)
print("jalan:", jalan_hasil)
print("remaining:", remaining)

jalan: BLOK M
remaining:    BLOK KV,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [114]:
def extract_and_remove_blok(text):
    m = re.search(BLOK_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

blok_hasil, remaining = extract_and_remove_blok(remaining)
print("blok:", blok_hasil)
print("remaining:", remaining)

blok: KV
remaining:    ,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [115]:
def extract_and_remove_kav(text):
    m = re.search(KAV_PATTERN, text)
    if not m:
        return None, text
    value = m.group(1)
    new_text = text[:m.start()] + text[m.end():]
    return value, new_text

kav_hasil, remaining = extract_and_remove_kav(remaining)
print("kav:", kav_hasil)
print("remaining:", remaining)

kav: None
remaining:    ,   JAKARTA - INDONESIA , JAKARTA SELATAN


In [116]:
# Clean leftover function

def clean_leftover(s):
    s = re.sub(NO_PATTERN, "", s)
    s = re.sub(FLOOR_PATTERN, "", s)
    s = re.sub(BLOK_PATTERN, "", s)
    s = re.sub(KAV_PATTERN, "", s)
    for p in RT_RW_PATTERNS:
        s = re.sub(p, "", s)
    return re.sub(r"\s+", " ", s).strip(" .,-")

print(clean_leftover(remaining))

JAKARTA - INDONESIA , JAKARTA SELATAN


In [121]:
# Nama yang eksis SEKALIGUS sebagai provinsi DAN sebagai desa/kelurahan di tempat lain.
# Match ke nama-nama ini WAJIB diverifikasi lebih lanjut, gak boleh diterima mentah-mentah.
AMBIGUOUS_PROVINCE_NAMES = {"ACEH", "PAPUA", "JAWA", "BALI"}

def match_wilayah_cascade_safe(leftover_text):
    tokens = tokenize(expand_abbreviations(leftover_text))
    result = {
        "provinsi": None, "kode_prov": None, "score_prov": 0,
        "kabkota": None, "kode_kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "kode_kec": None, "score_kec": 0,
        "desa": None, "kode_desa": None, "score_desa": 0,
        "ambiguous_province_flag": False,
    }

    # --- LEVEL 1: PROVINSI ---
    nama_p, kode_p = exact_match_safe(tokens, df_prov, min_len=4)
    score_p = 100
    if not nama_p:
        nama_p, kode_p, score_p = fuzzy_match_one(leftover_text, df_prov)
    if not nama_p:
        print("provinsi: TIDAK KETEMU -> stop")
        return result

    # --- GUARD: kalau provinsi yang ketemu itu nama ambigu, wajib ada konfirmasi ---
    is_ambiguous_name = any(w == nama_p or w in nama_p.split() for w in AMBIGUOUS_PROVINCE_NAMES)
    if is_ambiguous_name:
        print(f"provinsi '{nama_p}' masuk daftar AMBIGU -> perlu konfirmasi kab/kota")
        kandidat_kab_cek = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")]
        nama_k_cek, kode_k_cek = exact_match_safe(tokens, kandidat_kab_cek, min_len=4)
        if not nama_k_cek:
            print(f"  -> TIDAK ada kab/kota di provinsi '{nama_p}' yang match di teks -> KEMUNGKINAN SALAH, ini bisa jadi nama desa, bukan provinsi")
            result["ambiguous_province_flag"] = True
            # jangan langsung nyerah -- coba cari desa/kecamatan bernama sama di provinsi LAIN dulu
            nama_d_alt, kode_d_alt = exact_match_safe(tokens, df_desa, min_len=4)
            if nama_d_alt and nama_d_alt == nama_p:
                # exact match nama desa = nama yang tadinya dikira provinsi -> ambil ini, bukan provinsi
                print(f"  -> ditemukan DESA dengan nama sama: '{nama_d_alt}' (kode={kode_d_alt}) -> pakai ini, override provinsi tadi")
                parts = kode_d_alt.split(".")
                result["desa"], result["kode_desa"], result["score_desa"] = nama_d_alt, kode_d_alt, 90  # confidence diturunin krn butuh disambiguasi
                row_kec = df_kec[df_kec["kode"] == ".".join(parts[:3])]
                row_kab = df_kabkota[df_kabkota["kode"] == ".".join(parts[:2])]
                row_prov = df_prov[df_prov["kode"] == parts[0]]
                if not row_kec.empty:
                    result["kecamatan"], result["kode_kec"], result["score_kec"] = row_kec.iloc[0]["nama"], ".".join(parts[:3]), 90
                if not row_kab.empty:
                    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = row_kab.iloc[0]["nama"], ".".join(parts[:2]), 90
                if not row_prov.empty:
                    result["provinsi"], result["kode_prov"], result["score_prov"] = row_prov.iloc[0]["nama"], parts[0], 90
                result["ambiguous_province_flag"] = True  # tetap flag walau resolved, buat jaga-jaga
                return result
            else:
                print(f"  -> tidak ketemu alternatif juga, tetap pakai provinsi '{nama_p}' TAPI flag manual review")
        else:
            print(f"  -> terkonfirmasi, ada kab/kota '{nama_k_cek}' yang match -> provinsi '{nama_p}' valid")

    result["provinsi"], result["kode_prov"], result["score_prov"] = nama_p, kode_p, score_p

    # --- LEVEL 2: KAB/KOTA (lanjut normal) ---
    kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_p + ".")]
    nama_k, kode_k = exact_match_safe(tokens, kandidat_kab, min_len=4)
    score_k = 100
    if not nama_k:
        nama_k, kode_k, score_k = fuzzy_match_one(leftover_text, kandidat_kab)
    if not nama_k:
        return result
    result["kabkota"], result["kode_kabkota"], result["score_kabkota"] = nama_k, kode_k, score_k

    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_k + ".")]
    nama_c, kode_c = exact_match_safe(tokens, kandidat_kec, min_len=4)
    score_c = 100
    if not nama_c:
        nama_c, kode_c, score_c = fuzzy_match_one(leftover_text, kandidat_kec)
    if not nama_c:
        return result
    result["kecamatan"], result["kode_kec"], result["score_kec"] = nama_c, kode_c, score_c

    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_c + ".")]
    nama_d, kode_d = exact_match_safe(tokens, kandidat_desa, min_len=4)
    score_d = 100
    if not nama_d:
        nama_d, kode_d, score_d = fuzzy_match_one(leftover_text, kandidat_desa)
    if nama_d:
        result["desa"], result["kode_desa"], result["score_desa"] = nama_d, kode_d, score_d

    return result


# test kasus Desa Aceh (tanpa nyebut Sumatera Selatan)
test_case = "DESA ACEH KEC PAJAR BULAN KAB LAHAT"
hasil = match_wilayah_cascade_safe(test_case)
print("\nHASIL:", hasil)

provinsi 'ACEH' masuk daftar AMBIGU -> perlu konfirmasi kab/kota
  -> TIDAK ada kab/kota di provinsi 'ACEH' yang match di teks -> KEMUNGKINAN SALAH, ini bisa jadi nama desa, bukan provinsi
  -> tidak ketemu alternatif juga, tetap pakai provinsi 'ACEH' TAPI flag manual review

HASIL: {'provinsi': 'ACEH', 'kode_prov': '11', 'score_prov': 100, 'kabkota': None, 'kode_kabkota': None, 'score_kabkota': 0, 'kecamatan': None, 'kode_kec': None, 'score_kec': 0, 'desa': None, 'kode_desa': None, 'score_desa': 0, 'ambiguous_province_flag': True}


# SEGMENT ADDRESS

In [ ]:
def segment_address(alamat_upper):
    raw = [s.strip() for s in re.split(r"[,\n|]", alamat_upper) if s.strip()]
    building, street, wilayah_seg = [], [], []
    for seg in raw:
        if re.search(BUILDING_PATTERN, seg):
            building.append(seg)
        elif re.search(STREET_PATTERN, seg):
            street.append(seg)
        else:
            wilayah_seg.append(seg)
    return {"building": building, "street": street, "wilayah": wilayah_seg}

seg = segment_address(test_addr)
print(seg)

{'building': ['RT 02RW 07 GD SENTRAYA MARYA JAKARTA 12950- INDONESIA'], 'street': ['JL BLOK M NO. 12 LT 17 BLOK KV'], 'wilayah': ['JAKARTA SELATAN']}


In [89]:
BUILDING_NAME_PATTERN = re.compile(
    r"\b(?:GEDUNG|GD|GDG|GED|MENARA|MNR|TOWER|TWR|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b\.?\s*"
    r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,2})"  # maksimal 3 kata (kata pertama + 2 tambahan)
)

def extract_building_validated(text):
    m = BUILDING_NAME_PATTERN.search(text)
    if not m:
        return None
    raw_name = m.group(1).strip(" .,-")
    if not raw_name:
        return None

    # cross-check: kalau ada kata di raw_name yang match EXACT ke provinsi/kabkota,
    # potong capture SEBELUM kata itu (karena itu udah masuk wilayah, bukan nama gedung)
    words = raw_name.split()
    known_wilayah_tokens = set(tokenize(" ".join(df_prov["nama_core"]))) | set(tokenize(" ".join(df_kabkota["nama_core"])))

    cut_index = len(words)
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:  # min_len biar gak kena kata pendek umum
            cut_index = i
            break

    final_words = words[:cut_index]
    result = " ".join(final_words).strip(" .,-")
    return result if result else None

print(extract_building_validated(test_addr))

SENTRAYA MARYA


In [93]:
STREET_NAME_PATTERN = re.compile(
    r"\b(?:JALAN|JL|JLN)\b\.?\s*"
    r"([A-Z0-9][A-Z0-9\-]*(?:\s+[A-Z0-9][A-Z0-9\-]*){0,3})"  # maks 4 kata
)

def extract_street_validated(text):
    m = STREET_NAME_PATTERN.search(text)
    if not m:
        return None
    raw_name = m.group(1).strip(" .,-")
    if not raw_name:
        return None

    words = raw_name.split()
    known_wilayah_tokens = set(tokenize(" ".join(df_prov["nama_core"]))) | set(tokenize(" ".join(df_kabkota["nama_core"])))

    # potong di kata pertama yang exact match ke provinsi/kabkota
    cut_index = len(words)
    for i, w in enumerate(words):
        if w in known_wilayah_tokens and len(w) >= 4:
            cut_index = i
            break

    # potong juga di kata struktural (NO/LT/RT/dst) kalau kebawa ke dalam capture
    struktural_words = {"NO", "NOMOR", "NMR", "LT", "LANTAI", "LTAI", "FLOOR", "FLR", "RT", "RW"}
    for i, w in enumerate(words[:cut_index]):
        if w in struktural_words:
            cut_index = min(cut_index, i)
            break

    final = " ".join(words[:cut_index]).strip(" .,-")
    return final if final else None

# WAJIB pastiin test_addr sudah di-set ke alamat yang mau di-test SEBELUM baris ini
test_addr = "JL BLOK M NO. 12 LT 17 BLOK KV, RT 02RW 07 GD SENTRAYA MARYA JAKARTA 12950- INDONESIA , JAKARTA SELATAN".upper()

print(extract_street_validated(test_addr))

BLOK M


In [22]:
wilayah_text_raw = " , ".join(seg["wilayah"])
wilayah_text = expand_abbreviations(wilayah_text_raw)
print("raw:", wilayah_text_raw)
print("expanded:", wilayah_text)

raw: JAKARTA 12950 - INDONESIA , JAKARTA SELATAN
expanded: JAKARTA 12950 - INDONESIA , JAKARTA SELATAN


In [72]:
def sequence_in_tokens(cand_tokens, text_tokens):
    n, m = len(cand_tokens), len(text_tokens)
    if n == 0 or n > m:
        return False
    for i in range(m - n + 1):
        if text_tokens[i:i+n] == cand_tokens:
            return True
    return False

# test kasus bug ONESIA vs INDONESIA
print(sequence_in_tokens(["ONESIA"], tokenize("INDONESIA")))  # harus False sekarang
print(sequence_in_tokens(["JAKARTA", "SELATAN"], text_tokens))  # harus True

False
True


In [73]:
def exact_match_safe(text_tokens, candidates_df, min_len=4, name_col="nama_core"):
    text_tokens_set = set(text_tokens)
    cand_sorted = candidates_df.assign(_len=candidates_df[name_col].str.len()).sort_values("_len", ascending=False)
    for _, row in cand_sorted.iterrows():
        core = row[name_col]
        if not core or len(core) < min_len:
            continue
        cand_tokens = tokenize(core)
        if not cand_tokens:
            continue
        if sequence_in_tokens(cand_tokens, text_tokens):
            return row["nama"], row["kode"]
        cand_nospace = "".join(cand_tokens)
        if len(cand_tokens) > 1 and cand_nospace in text_tokens_set and len(cand_nospace) >= min_len:
            return row["nama"], row["kode"]
    return None, None

nama_kab, kode_kab = exact_match_safe(text_tokens, df_kabkota, min_len=4)
print(nama_kab, kode_kab)

KOTA ADMINISTRASI JAKARTA SELATAN 31.74


In [26]:
FUZZY_THRESHOLD = 95

def fuzzy_match_one(text, candidates_df, score_cutoff=FUZZY_THRESHOLD):
    if candidates_df.empty:
        return None, None, 0
    match = process.extractOne(text, candidates_df["nama"], scorer=fuzz.token_set_ratio, score_cutoff=score_cutoff)
    if not match:
        return None, None, 0
    nama, score, idx = match
    row = candidates_df.loc[idx]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    return row["nama"], row["kode"], score

print(fuzzy_match_one(wilayah_text, df_prov))

(None, None, 0)


In [27]:
if nama_kab:
    kode_prov_derived = kode_kab.split(".")[0]
    row_prov = df_prov[df_prov["kode"] == kode_prov_derived]
    print(row_prov[["nama", "kode"]])

                             nama kode
10  DAERAH KHUSUS IBUKOTA JAKARTA   31


In [28]:
if nama_kab:
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_kab + ".")]
    nama_kec, kode_kec = exact_match_safe(text_tokens, kandidat_kec, min_len=5)
    print("kecamatan:", nama_kec, kode_kec)

    if nama_kec:
        kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_kec + ".")]
        nama_desa, kode_desa = exact_match_safe(text_tokens, kandidat_desa, min_len=6)
        print("desa:", nama_desa, kode_desa)

kecamatan: None None


In [29]:
if kodepos_hasil:
    kandidat_kode_postcode = kodepos_df.loc[kodepos_df["kodepos"] == str(kodepos_hasil), "kode"].tolist()
    print(f"kandidat dari kodepos {kodepos_hasil}: {len(kandidat_kode_postcode)} kode")
    pool_desa = df_desa[df_desa["kode"].isin(kandidat_kode_postcode)]
    nama_d_pc, kode_d_pc = exact_match_safe(text_tokens, pool_desa, min_len=4)
    print("desa dari pool kodepos:", nama_d_pc, kode_d_pc)

kandidat dari kodepos 12950: 1 kode
desa dari pool kodepos: None None
